# Predicción de enlaces
## Star Wars II: Attack of the Clones

Sea $G=(V,E)$ una red simple no dirigida, donde cada vértice $v\in V$ representa un personaje y cada arista $(u,v)\in E$ representa una coaparición en al menos una escena. La red original puede contener información adicional, como pesos o atributos, pero para este ejercicio se usa la versión binaria del grafo: dos personajes están conectados si coaparecen, independientemente del número de escenas compartidas.

El problema de predicción de enlaces consiste en asignar un puntaje $s(u,v)$ a cada par no observado $(u,v)
otin E$. La interpretación del puntaje no es probabilística en sentido estricto; funciona como una medida estructural de plausibilidad: pares con mayor $s(u,v)$ son candidatos más razonables a convertirse en enlaces si la red creciera manteniendo sus patrones actuales de conectividad.

En términos narrativos, una predicción alta sugiere que dos personajes ocupan posiciones compatibles dentro de la estructura de coapariciones: pueden compartir intermediarios, pertenecer al mismo bloque de escenas o estar conectados indirectamente por personajes centrales.


In [1]:
import html
import math
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

import networkx as nx
from IPython.display import HTML, display


In [2]:
def find_repo_file(relative_path):
    relative_path = Path(relative_path)
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontro {relative_path} subiendo desde {Path.cwd()}")


def ascii_text(value):
    value = unicodedata.normalize("NFD", str(value))
    return "".join(ch for ch in value if unicodedata.category(ch) != "Mn")


def load_gexf_simple_undirected(path):
    """Carga nodos/aristas de un GEXF de Gephi y conserva las etiquetas legibles."""
    namespace = {"g": "http://www.gexf.net/1.2draft"}
    root = ET.parse(path).getroot()

    labels = {
        node.attrib["id"]: ascii_text(node.attrib.get("label", node.attrib["id"]))
        for node in root.findall(".//g:node", namespace)
    }

    G = nx.Graph()
    G.add_nodes_from(labels.values())
    for edge in root.findall(".//g:edge", namespace):
        u = labels[edge.attrib["source"]]
        v = labels[edge.attrib["target"]]
        if u != v:
            G.add_edge(u, v)
    return G

movie_id = 774
gexf_path = find_repo_file(Path("data/dataverse/gexf") / f"{movie_id}.gexf")
G1 = load_gexf_simple_undirected(gexf_path)
print("loaded", movie_id, "| Nodos:", G1.number_of_nodes(), "| Aristas:", G1.number_of_edges())


loaded 774 | Nodos: 47 | Aristas: 148


In [3]:
def show_table(rows, columns=None, float_digits=4):
    if not rows:
        display(HTML("<em>Sin datos.</em>"))
        return

    if columns is None:
        columns = list(rows[0].keys())

    def fmt(value):
        if isinstance(value, float):
            return f"{value:.{float_digits}f}"
        return html.escape(str(value))

    header = "".join(f"<th>{html.escape(str(col))}</th>" for col in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{fmt(row.get(col, ''))}</td>" for col in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"""
    <table style="border-collapse:collapse; font-size:14px; max-width:100%;">
      <thead><tr style="background:#f2f2f2;">{header}</tr></thead>
      <tbody>{body}</tbody>
    </table>
    <style>
      table td, table th {{ border:1px solid #ddd; padding:5px 8px; vertical-align:top; }}
      table th {{ text-align:left; }}
    </style>
    """))


def graph_summary(G):
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    communities, _ = modularity_partition(G)
    return [{
        "nodos": G.number_of_nodes(),
        "aristas": G.number_of_edges(),
        "densidad": nx.density(G),
        "componentes": len(components),
        "tamano_componente_mayor": len(components[0]) if components else 0,
        "comunidades_modularidad": len(communities),
        "pares_sin_arista": len(list(nx.non_edges(G)))
    }]


def top_degrees(G, k=10):
    return [
        {"vertice": node, "grado": degree}
        for node, degree in sorted(G.degree, key=lambda item: item[1], reverse=True)[:k]
    ]


def modularity_partition(G):
    communities = list(nx.algorithms.community.greedy_modularity_communities(G))
    communities = [sorted(list(c), key=str) for c in communities]
    communities.sort(key=lambda c: (-len(c), str(c[0]) if c else ""))
    community_of = {}
    for idx, community in enumerate(communities):
        for node in community:
            community_of[node] = idx
    return communities, community_of


def prediction_table(G, method, k=10):
    """Devuelve las k aristas no observadas con mayor puntaje."""
    ebunch = list(nx.non_edges(G))
    communities, community_of = modularity_partition(G)
    if method == "PAC":
        rows = nx.preferential_attachment(G, ebunch)
    elif method == "AAC":
        rows = nx.adamic_adar_index(G, ebunch)
    elif method == "JAC":
        rows = nx.jaccard_coefficient(G, ebunch)
    else:
        raise ValueError("method debe ser PAC, AAC o JAC")

    rows = sorted(rows, key=lambda row: (row[2], str(row[0]), str(row[1])), reverse=True)[:k]
    out = []
    for u, v, score in rows:
        cu = community_of.get(u, -1)
        cv = community_of.get(v, -1)
        out.append({
            "u": u,
            "v": v,
            "puntaje": score,
            "comunidad_u": cu,
            "comunidad_v": cv,
            "tipo_modularidad": "intra" if cu == cv else "inter",
            "vecinos_comunes": ", ".join(sorted(nx.common_neighbors(G, u, v)))
        })
    return out


def _scale_positions(pos, width, height, margin):
    return _scale_positions_box(pos, width, height, margin, margin, margin, margin)


def _scale_positions_box(pos, width, height, left, right, top, bottom):
    xs = [xy[0] for xy in pos.values()]
    ys = [xy[1] for xy in pos.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    span_x = max(max_x - min_x, 1e-9)
    span_y = max(max_y - min_y, 1e-9)
    return {
        node: (
            left + (xy[0] - min_x) / span_x * (width - left - right),
            top + (xy[1] - min_y) / span_y * (height - top - bottom),
        )
        for node, xy in pos.items()
    }


def modularity_seed_layout(G):
    communities, community_of = modularity_partition(G)
    pos = {}
    n_com = max(len(communities), 1)
    for ci, community in enumerate(communities):
        center_angle = 2 * math.pi * ci / n_com
        cx = 2.8 * math.cos(center_angle)
        cy = 2.8 * math.sin(center_angle)
        inner_n = max(len(community), 1)
        inner_radius = 0.42 + 0.055 * inner_n
        for j, node in enumerate(community):
            angle = 2 * math.pi * j / inner_n + 0.37 * ci
            pos[node] = (cx + inner_radius * math.cos(angle), cy + inner_radius * math.sin(angle))
    return pos, communities, community_of


def force_layout(G, iterations=180):
    pos, communities, community_of = modularity_seed_layout(G)
    nodes = list(G.nodes())
    n = max(len(nodes), 1)
    area = 32.0
    k = math.sqrt(area / n)
    edges = list(G.edges())

    for step in range(iterations):
        disp = {node: [0.0, 0.0] for node in nodes}
        temp = 0.22 * (1 - step / iterations) + 0.018

        for i, u in enumerate(nodes):
            ux, uy = pos[u]
            for v in nodes[i + 1:]:
                vx, vy = pos[v]
                dx = ux - vx
                dy = uy - vy
                dist = math.sqrt(dx * dx + dy * dy) + 1e-6
                force = (k * k) / dist
                fx = dx / dist * force
                fy = dy / dist * force
                disp[u][0] += fx
                disp[u][1] += fy
                disp[v][0] -= fx
                disp[v][1] -= fy

        for u, v in edges:
            ux, uy = pos[u]
            vx, vy = pos[v]
            dx = ux - vx
            dy = uy - vy
            dist = math.sqrt(dx * dx + dy * dy) + 1e-6
            same = community_of.get(u) == community_of.get(v)
            strength = 1.55 if same else 0.75
            force = strength * (dist * dist) / k
            fx = dx / dist * force
            fy = dy / dist * force
            disp[u][0] -= fx
            disp[u][1] -= fy
            disp[v][0] += fx
            disp[v][1] += fy

        for node in nodes:
            dx, dy = disp[node]
            length = math.sqrt(dx * dx + dy * dy) + 1e-6
            x, y = pos[node]
            pos[node] = (x + dx / length * min(length, temp), y + dy / length * min(length, temp))
    return pos, communities, community_of


def show_network_svg(G, title="Red", predicted=None, width=980, height=780, label_mode="auto"):
    predicted = predicted or []
    raw_pos, communities, community_of = force_layout(G)
    pos = _scale_positions_box(raw_pos, width, height, left=78, right=54, top=170, bottom=52)
    degrees = dict(G.degree())
    max_degree = max(degrees.values()) if degrees else 1
    palette = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#76b7b2", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab"]

    predicted_nodes = {node for row in predicted for node in (row["u"], row["v"])}
    top_label_nodes = {node for node, _ in sorted(G.degree, key=lambda item: item[1], reverse=True)[:10]}
    subtitle = f"{G.number_of_nodes()} nodos | {G.number_of_edges()} aristas observadas | {len(communities)} comunidades por modularidad"

    community_parts = []
    for ci, community in enumerate(communities):
        xs = [pos[node][0] for node in community]
        ys = [pos[node][1] for node in community]
        if not xs:
            continue
        cx = sum(xs) / len(xs)
        cy = sum(ys) / len(ys)
        radius = max([math.sqrt((x - cx) ** 2 + (y - cy) ** 2) for x, y in zip(xs, ys)] + [22]) + 40
        color = palette[ci % len(palette)]
        community_parts.append(
            f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{radius:.1f}" fill="{color}" opacity="0.075" stroke="{color}" stroke-width="1.3" stroke-dasharray="5 5" />'
        )
        community_parts.append(
            f'<text x="{cx:.1f}" y="{max(cy - radius + 18, 158):.1f}" font-size="12" text-anchor="middle" font-family="Arial" fill="{color}">C{ci} ({len(community)} nodos)</text>'
        )

    edge_parts = []
    for u, v in G.edges():
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        same = community_of.get(u) == community_of.get(v)
        stroke = "#a9b1bb" if same else "#7d8793"
        opacity = "0.42" if same else "0.68"
        width_line = "1.0" if same else "1.6"
        edge_parts.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{stroke}" stroke-width="{width_line}" opacity="{opacity}" />'
        )

    pred_parts = []
    if predicted:
        scores = [row["puntaje"] for row in predicted]
        min_s, max_s = min(scores), max(scores)
    else:
        min_s, max_s = 0, 1
    for row in predicted:
        u, v = row["u"], row["v"]
        if u not in pos or v not in pos:
            continue
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        score = row["puntaje"]
        same = community_of.get(u) == community_of.get(v)
        color = palette[community_of.get(u, 0) % len(palette)] if same else "#d62728"
        width_line = 2.4 + 3.2 * ((score - min_s) / (max_s - min_s + 1e-9))
        pred_parts.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="{width_line:.1f}" opacity="0.97">'
            f'<title>{html.escape(str(u))} - {html.escape(str(v))}: {score:.4f} | {"intra-comunidad" if same else "inter-comunidad"}</title></line>'
        )

    node_parts = []
    label_parts = []
    for node in G.nodes():
        x, y = pos[node]
        r = 7 + 14 * degrees[node] / max_degree
        color = palette[community_of.get(node, 0) % len(palette)]
        node_parts.append(
            f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" fill="{color}" stroke="#222" stroke-width="1.1">'
            f'<title>{html.escape(str(node))} | grado {degrees[node]} | comunidad C{community_of.get(node, -1)}</title></circle>'
        )
        should_label = label_mode == "all" or node in predicted_nodes or node in top_label_nodes
        if should_label:
            label = html.escape(str(node))
            label_parts.append(
                f'<text x="{x:.1f}" y="{y-r-5:.1f}" font-size="10" text-anchor="middle" font-family="Arial, sans-serif" fill="#1b1b1b">{label}</text>'
            )

    community_legend = []
    for ci, community in enumerate(communities[:6]):
        color = palette[ci % len(palette)]
        x = 565 + (ci % 3) * 122
        y = 88 + (ci // 3) * 26
        community_legend.append(
            f'<circle cx="{x:.1f}" cy="{y:.1f}" r="6" fill="{color}" stroke="#222" stroke-width="0.6" />'
            f'<text x="{x+12:.1f}" y="{y+4:.1f}" font-size="12" font-family="Arial" fill="#333">C{ci}: {len(community)}</text>'
        )

    svg = f"""
    <div style="max-width:{width}px; overflow-x:auto; font-family:Arial, sans-serif;">
      <svg viewBox="0 0 {width} {height}" width="100%" height="auto" role="img" aria-label="{html.escape(title)}">
        <title>{html.escape(title)}</title>
        <desc>{html.escape(subtitle)}. Los colores indican comunidades de modularidad; las lineas gruesas son enlaces predichos.</desc>
        <rect width="100%" height="100%" fill="#ffffff" />

        <text x="24" y="32" font-size="22" font-family="Arial, sans-serif" font-weight="700" fill="#111">{html.escape(title)}</text>
        <text x="24" y="56" font-size="13" font-family="Arial, sans-serif" fill="#555">{html.escape(subtitle)}</text>

        <rect x="20" y="70" width="930" height="54" rx="8" fill="#f8f9fb" stroke="#d6dbe1" stroke-width="1" />
        <text x="34" y="92" font-size="13" font-family="Arial" font-weight="700" fill="#222">Leyenda</text>
        <line x1="34" y1="110" x2="86" y2="110" stroke="#a9b1bb" stroke-width="1.6" opacity="0.8" />
        <text x="96" y="114" font-size="12" font-family="Arial" fill="#333">Coaparición</text>
        <line x1="220" y1="110" x2="272" y2="110" stroke="#4e79a7" stroke-width="4" />
        <text x="282" y="114" font-size="12" font-family="Arial" fill="#333">Predicción dentro de su comunidad</text>
        <line x1="545" y1="110" x2="597" y2="110" stroke="#d62728" stroke-width="4" />
        <text x="607" y="114" font-size="12" font-family="Arial" fill="#333">Predicción fuera de su comunidad</text>
        <circle cx="780" cy="110" r="5" fill="#4e79a7" stroke="#222" stroke-width="0.8" />
        <circle cx="798" cy="110" r="10" fill="#4e79a7" stroke="#222" stroke-width="0.8" />
        <text x="814" y="114" font-size="12" font-family="Arial" fill="#333">Tamaño del nodo = grado</text>

        <g>{''.join(community_parts)}</g>
        <g>{''.join(edge_parts)}</g>
        <g>{''.join(pred_parts)}</g>
        <g>{''.join(node_parts)}</g>
        <g>{''.join(label_parts)}</g>
      </svg>
    </div>
    """
    display(HTML(svg))


## Análisis de la red

La red tiene $|V|=47$ vértices y $|E|=148$ aristas. El número máximo de aristas posibles en un grafo simple no dirigido con 47 vértices es $\binom{47}{2}=1081$, por lo que la densidad es

$$
\rho(G)=\frac{2|E|}{|V|(|V|-1)}\approx 0.137.
$$

Esta densidad indica que la red no es completa, pero tiene un núcleo narrativo suficientemente conectado. Los vértices de mayor grado son PADME con 30 conexiones, OBI-WAN con 29, ANAKIN con 24, YODA con 14, MACE WINDU con 13, y luego PALPATINE y JAR JAR con 11. El grado $k_v$ mide cuántos personajes coaparecen directamente con $v$, por lo que estos nodos funcionan como articuladores del relato.

La predicción se calcula sobre 933 pares sin arista. Esos pares son el conjunto

$$
\overline{E}=\{(u,v): u,v\in V,\ u\neq v,\ (u,v)\notin E\}.
$$

Para interpretar los resultados se usa también una partición por modularidad. La modularidad evalúa qué tan concentradas están las aristas dentro de comunidades comparado con un modelo nulo que conserva grados. En forma estándar,

$$
Q=\frac{1}{2m}\sum_{i,j}\left(A_{ij}-\frac{k_i k_j}{2m}\right)\delta(c_i,c_j),
$$

donde $A_{ij}$ es la matriz de adyacencia, $k_i$ es el grado de $i$, $m=|E|$, $c_i$ es la comunidad de $i$ y $\delta(c_i,c_j)=1$ si ambos nodos pertenecen a la misma comunidad. En la notebook se usa `greedy_modularity_communities`, que construye comunidades buscando aumentar $Q$ de forma aglomerativa. Esta partición permite clasificar cada predicción como intra-comunidad o inter-comunidad.


In [4]:
show_table(graph_summary(G1))
show_table(top_degrees(G1, k=10))


nodos,aristas,densidad,componentes,tamano_componente_mayor,comunidades_modularidad,pares_sin_arista
47,148,0.1369,1,47,4,933


vertice,grado
PADME,30
OBI-WAN,29
ANAKIN,24
YODA,14
MACE WINDU,13
JAR JAR,11
PALPATINE,11
MAS AMEDDA,10
SENATOR ASK AAK,10
BAIL ORGANA,9


## Figura de la red base

La figura usa un layout tipo fuerza inicializado con comunidades de modularidad. Primero se detectan comunidades maximizando modularidad de forma aproximada; después se colocan los nodos de la misma comunidad cerca entre sí y se aplica una dinámica de fuerzas repulsivas y atractivas para separar visualmente los bloques.

El color del nodo representa su comunidad. El tamaño del nodo representa el grado $k_v$, es decir, el número de coapariciones directas del personaje. En las figuras de PAC, AAC y JAC, una predicción dentro de comunidad conserva el color de esa comunidad; una predicción fuera de comunidad aparece en rojo.

Esta codificación visual permite distinguir dos fenómenos: cierre local de triángulos dentro de comunidades y formación potencial de puentes entre comunidades. En análisis de redes, esta diferencia es importante porque un enlace intra-comunidad refuerza cohesión local, mientras que un enlace inter-comunidad puede reducir distancias geodésicas y aumentar la integración global de la red.


In [5]:
show_network_svg(G1, title="Red base con comunidades de modularidad")


## Resumen de las tres técnicas

Para cada par no observado $(u,v)\in\overline{E}$ se calculan tres puntajes de predicción. Sea $\Gamma(u)$ el conjunto de vecinos de $u$ y sea $k_u=|\Gamma(u)|$ su grado.

Emparejamiento Preferencial (PAC) se define como

$$
s_{PAC}(u,v)=k_u k_v.
$$

Este índice se basa en un mecanismo de acumulación: nodos con mayor grado tienden a adquirir más enlaces. Es una medida global-local, porque solo usa los grados de los extremos y no requiere vecinos comunes.

Adamic-Adar (AAC) se define como

$$
s_{AAC}(u,v)=\sum_{w\in\Gamma(u)\cap\Gamma(v)}\frac{1}{\log k_w}.
$$

Este índice usa vecinos comunes, pero penaliza vecinos demasiado populares. Un intermediario con grado bajo aporta más evidencia que un intermediario conectado con casi toda la red.

Jaccard (JAC) se define como

$$
s_{JAC}(u,v)=\frac{|\Gamma(u)\cap\Gamma(v)|}{|\Gamma(u)\cup\Gamma(v)|}.
$$

Este índice mide similitud relativa de vecindarios. Puede ser alto incluso para nodos periféricos si sus vecindarios son casi idénticos.

La diferencia central es que PAC privilegia popularidad, AAC privilegia vecinos comunes informativos y JAC privilegia similitud proporcional. Por eso no deben interpretarse como variantes equivalentes, sino como hipótesis estructurales distintas sobre cómo podría crecer la red.


## Usando Emparejamiento Preferencial (PAC)

PAC ordena los pares no observados según el producto de grados. En esta red aparecen ANAKIN--PALPATINE con puntaje 264, ANAKIN--SENATOR ASK AAK con 240, ANAKIN--MAS AMEDDA con 240 y ANAKIN--BAIL ORGANA con 216. El patrón es claro: ANAKIN tiene grado alto y se combina con personajes que también están relativamente conectados dentro del bloque político/Jedi.

Formalmente, PAC no evalúa $\Gamma(u)\cap\Gamma(v)$. Por eso puede asignar puntajes altos a pares sin evidencia de vecindario compartido. En términos de modularidad, esto significa que PAC puede producir tanto enlaces intra-comunidad como inter-comunidad. Los enlaces intra-comunidad refuerzan grupos ya densos; los enlaces inter-comunidad aparecen porque ambos extremos tienen grado alto, no necesariamente porque compartan contexto local.

En esta red narrativa, PAC captura bien la centralidad de los protagonistas. Sin embargo, su sesgo hacia el grado implica que puede sobreestimar coapariciones entre personajes populares que pertenecen a escenas o subtramas distintas. Por eso PAC es adecuado para estudiar expansión del núcleo, pero menos preciso para identificar cierres locales específicos.


In [6]:
pac = prediction_table(G1, "PAC", k=10)
show_table(pac)
show_network_svg(G1, title="Enlaces predichos con PAC", predicted=pac[:8])


u,v,puntaje,comunidad_u,comunidad_v,tipo_modularidad,vecinos_comunes
PALPATINE,ANAKIN,264,1,0,inter,"JAR JAR, MACE WINDU, OBI-WAN, PADME, YODA"
SENATOR ASK AAK,ANAKIN,240,1,0,inter,"JAR JAR, MACE WINDU, OBI-WAN, PADME, YODA"
MAS AMEDDA,ANAKIN,240,1,0,inter,"JAR JAR, MACE WINDU, OBI-WAN, PADME, YODA"
BAIL ORGANA,ANAKIN,216,1,0,inter,"JAR JAR, MACE WINDU, OBI-WAN, PADME, YODA"
ORN FREE TAA,OBI-WAN,174,1,2,inter,"JAR JAR, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
AMIDALA,OBI-WAN,174,1,2,inter,"CAPTAIN TYPHO, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
PADME,TAUN WE,150,0,2,inter,"JANGO FETT, OBI-WAN"
OWEN,OBI-WAN,145,0,2,inter,"ANAKIN, PADME"
KI-ADI-MUNDI,OBI-WAN,145,1,2,inter,"BAIL ORGANA, MACE WINDU, PADME, PALPATINE, YODA"
BERU,OBI-WAN,145,0,2,inter,"ANAKIN, PADME"


## Usando el Coeficiente Adamic-Adar (AAC)

Adamic-Adar evalúa la evidencia aportada por vecinos comunes. El enlace más alto es AMIDALA--JAR JAR, con puntaje aproximado 2.696; comparten CAPTAIN TYPHO, MAS AMEDDA, ORN FREE TAA, PADME, PALPATINE y SENATOR ASK AAK. Estos vecinos comunes no solo conectan a los extremos, sino que ubican el par dentro de una región política coherente de la red.

La diferencia respecto a PAC es metodológica. AAC no premia simplemente que $u$ y $v$ tengan grado alto; premia que existan caminos de longitud dos $u-w-v$ y que los intermediarios $w$ no sean excesivamente genéricos. Por eso aparecen AMIDALA--OBI-WAN, ORN FREE TAA--YODA, OBI-WAN--ORN FREE TAA, MACE WINDU--ORN FREE TAA y BAIL ORGANA--ORN FREE TAA.

Desde la modularidad, AAC suele comportarse como un detector de cierre intra-comunidad. Cuando produce enlaces inter-comunidad, esos enlaces son más interpretables que en PAC porque existe soporte por vecinos comunes. En esta red, AAC identifica mejor relaciones latentes dentro del bloque Senado/Jedi y reduce el sesgo de los protagonistas de máximo grado.


In [7]:
aac = prediction_table(G1, "AAC", k=10)
show_table(aac)
show_network_svg(G1, title="Enlaces predichos con AAC", predicted=aac[:8])


u,v,puntaje,comunidad_u,comunidad_v,tipo_modularidad,vecinos_comunes
AMIDALA,JAR JAR,2.6959,1,1,intra,"CAPTAIN TYPHO, MAS AMEDDA, ORN FREE TAA, PADME, PALPATINE, SENATOR ASK AAK"
AMIDALA,OBI-WAN,2.1377,1,2,inter,"CAPTAIN TYPHO, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
YODA,ORN FREE TAA,1.9967,1,1,intra,"JAR JAR, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
ORN FREE TAA,OBI-WAN,1.9967,1,2,inter,"JAR JAR, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
ORN FREE TAA,MACE WINDU,1.9967,1,1,intra,"JAR JAR, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
BAIL ORGANA,ORN FREE TAA,1.9967,1,1,intra,"JAR JAR, MAS AMEDDA, PADME, PALPATINE, SENATOR ASK AAK"
SENATOR ASK AAK,KI-ADI-MUNDI,1.9350,1,1,intra,"BAIL ORGANA, MACE WINDU, PADME, PALPATINE, YODA"
MAS AMEDDA,KI-ADI-MUNDI,1.9350,1,1,intra,"BAIL ORGANA, MACE WINDU, PADME, PALPATINE, YODA"
KI-ADI-MUNDI,OBI-WAN,1.9350,1,2,inter,"BAIL ORGANA, MACE WINDU, PADME, PALPATINE, YODA"
KI-ADI-MUNDI,JAR JAR,1.9350,1,1,intra,"BAIL ORGANA, MACE WINDU, PADME, PALPATINE, YODA"


## Usando el Coeficiente Jaccard (JAC)

Jaccard mide la similitud relativa entre vecindarios. En la red aparecen enlaces con puntaje 1.0 como SHMI--WATTO, JOCASTA NU--PK-4 y ELAN--ZAM WESSEL. Un puntaje igual a 1 implica que, para ese par, todos los vecinos observados son compartidos: $\Gamma(u)=\Gamma(v)$ en la parte relevante de la red.

Esta propiedad hace que Jaccard sea sensible a nodos de bajo grado. Si dos personajes periféricos comparten únicamente a ANAKIN, el cociente puede ser máximo aunque la evidencia absoluta sea pequeña. Por eso JAC no debe leerse como importancia global, sino como equivalencia local de posición estructural.

En términos de comunidades, JAC tiende a generar cierres dentro de microgrupos o subescenas. Es útil para detectar personajes que ocupan roles similares alrededor de un mismo centro, pero no necesariamente para detectar puentes narrativos de gran escala. Su lectura complementa a PAC y AAC porque muestra similitud de vecindario, no popularidad ni peso de intermediarios.


In [8]:
jac = prediction_table(G1, "JAC", k=10)
show_table(jac)
show_network_svg(G1, title="Enlaces predichos con JAC", predicted=jac[:8])


u,v,puntaje,comunidad_u,comunidad_v,tipo_modularidad,vecinos_comunes
ZAM WESSEL,ELAN,1.0000,2,2,intra,"ANAKIN, OBI-WAN"
WINDU,MACE,1.0000,1,1,intra,"MACE WINDU, OBI-WAN, YODA"
SHMI,WATTO,1.0000,0,0,intra,ANAKIN
JOCASTA NU,PK-4,1.0000,2,2,intra,OBI-WAN
THREEPIO,CLIEGG,0.7500,0,0,intra,"BERU, OWEN, PADME"
WINDU,CHILDREN,0.6667,1,1,intra,"OBI-WAN, YODA"
RYOO & POOJA,SIO BIBBLE,0.6667,0,0,intra,"ANAKIN, PADME"
QUEEN JAMILLIA,RYOO & POOJA,0.6667,0,0,intra,"ANAKIN, PADME"
CHILDREN,MACE,0.6667,1,1,intra,"OBI-WAN, YODA"
AMIDALA,JAR JAR,0.5455,1,1,intra,"CAPTAIN TYPHO, MAS AMEDDA, ORN FREE TAA, PADME, PALPATINE, SENATOR ASK AAK"


## Análisis específico de resultados

En esta red, PAC, AAC y JAC representan tres mecanismos distintos de formación de enlaces. PAC sigue un mecanismo de acumulación por grado: los personajes que ya coaparecen con muchos otros, como ANAKIN, OBI-WAN y PADME, concentran las predicciones más altas. Esto refleja una hipótesis de crecimiento tipo preferential attachment.

AAC ofrece una lectura más contextual. Al ponderar vecinos comunes poco frecuentes, identifica pares que comparten intermediarios narrativamente informativos. Por eso los enlaces asociados con AMIDALA, JAR JAR, ORN FREE TAA, YODA y OBI-WAN son relevantes: no solo tienen conexión indirecta, sino que esa conexión ocurre dentro de un bloque político/Jedi reconocible.

JAC detecta equivalencia local. Sus puntajes perfectos no significan máxima centralidad ni máxima importancia narrativa; significan que los extremos tienen vecindarios proporcionalmente idénticos o casi idénticos. Esto es especialmente visible en pares periféricos que orbitan a un mismo personaje central.

La modularidad permite evaluar el sentido estructural de cada predicción. Una predicción intra-comunidad aumenta la cohesión de un bloque narrativo; una predicción inter-comunidad funcionaría como puente entre bloques. En conjunto, AAC parece la técnica más estable para inferir coapariciones plausibles, PAC describe expansión del núcleo de protagonistas y JAC revela microestructuras de equivalencia local.
